In [ ]:
library(truncnorm)
library(ggplot2)

hybrid_pdf <- function(x, mu1, mu2, alpha) {
  mu <- (mu1 + mu2) / 2
  L <- min(mu1, mu2)
  U <- max(mu1, mu2)
  sigma1 <- sqrt(1/(2*alpha))
  sigma2 <- sqrt(1/(2*(1-alpha)))
  Phi1 <- function(z) pnorm(z, mean=mu, sd=sigma1)
  Phi2 <- function(z) pnorm(z, mean=mu, sd=sigma2)
  A1 <- Phi1(U) - Phi1(L)
  A2 <- 1 - (Phi2(U) - Phi2(L))
  Z <- A1 + A2
  dens <- numeric(length(x))
  idx1 <- (x >= L) & (x <= U)
  dens[idx1] <- dnorm(x[idx1], mean=mu, sd=sigma1) / Z
  idx2 <- !idx1
  dens[idx2] <- dnorm(x[idx2], mean=mu, sd=sigma2) / Z
  return(dens)
}
hybrid_sampler <- function(n, mu1, mu2, alpha) {
  mu <- (mu1 + mu2) / 2
  L <- min(mu1, mu2)
  U <- max(mu1, mu2)
  sigma1 <- sqrt(1/(2*alpha))
  sigma2 <- sqrt(1/(2*(1-alpha)))
  # For weights:
  Phi1_L <- pnorm(L, mean=mu, sd=sigma1)
  Phi1_U <- pnorm(U, mean=mu, sd=sigma1)
  Phi2_L <- pnorm(L, mean=mu, sd=sigma2)
  Phi2_U <- pnorm(U, mean=mu, sd=sigma2)
  A1 <- Phi1_U - Phi1_L
  A2 <- 1 - (Phi2_U - Phi2_L)
  Z <- A1 + A2
  prob1 <- A1 / Z
  # Bernoulli to decide regime
  B <- rbinom(n, size=1, prob=prob1)
  x <- numeric(n)
  # Inside [L, U]
  if(sum(B == 1) > 0)
    x[B == 1] <- rtruncnorm(sum(B == 1), a=L, b=U, mean=mu, sd=sigma1)
  # Outside [L, U]
  n2 <- sum(B == 0)
  if(n2 > 0) {
    left_tail_mass <- pnorm(L, mean=mu, sd=sigma2)
    right_tail_mass <- 1 - pnorm(U, mean=mu, sd=sigma2)
    total_tail_mass <- left_tail_mass + right_tail_mass
    # Which indices are outside?
    idx_outside <- which(B == 0)
    choose_left <- rbinom(n2, size=1, prob=left_tail_mass / total_tail_mass)
    # Assign left tail
    if(sum(choose_left) > 0)
      x[idx_outside[which(choose_left == 1)]] <- rtruncnorm(sum(choose_left), a=-Inf, b=L, mean=mu, sd=sigma2)
    # Assign right tail
    if(sum(choose_left == 0) > 0)
      x[idx_outside[which(choose_left == 0)]] <- rtruncnorm(sum(choose_left == 0), a=U, b=Inf, mean=mu, sd=sigma2)
  }
  return(x)
}
# Set parameters
alpha <- 0.05
mu1 <- -2
mu2 <- 1
set.seed(2024)
n <- 100000

samples <- hybrid_sampler(n, mu1, mu2, alpha)
x_grid <- seq(mu1 - 4, mu2 + 4, length.out=1000)
dens_grid <- hybrid_pdf(x_grid, mu1, mu2, alpha)
df <- data.frame(x = x_grid, dens = dens_grid)

ggplot() +
  geom_histogram(aes(x = samples, y = after_stat(density)), 
                 bins = 80, fill = "skyblue", alpha = 0.5, boundary = 0) +
  geom_line(data = df, aes(x = x, y = dens), color = "darkred", linewidth = 0.5) +
  labs(title = "Hybrid Normal Distribution: Empirical vs. True Density",
       x = "x", y = "Density") +
  theme_minimal(base_size = 15)


In [ ]:
library(truncnorm)
library(bbmle)

loglik_int_param <- function(theta1, theta2, alpha, data) {
  L <- min(theta1, theta2)
  U <- max(theta1, theta2)
  if (L >= U) return(-1e10) # always enforce

  mu <- (L + U) / 2
  sigma1 <- sqrt(1/(2*alpha))
  sigma2 <- sqrt(1/(2*(1 - alpha)))
  
  # Normalization constant
  Phi1_L <- pnorm(L, mean=mu, sd=sigma1)
  Phi1_U <- pnorm(U, mean=mu, sd=sigma1)
  Phi2_L <- pnorm(L, mean=mu, sd=sigma2)
  Phi2_U <- pnorm(U, mean=mu, sd=sigma2)
  Z <- 1 - (Phi2_U - Phi2_L) + (Phi1_U - Phi1_L)
  
  # Piecewise log-density
  inside <- (data >= L) & (data <= U)
  logphi1 <- dnorm(data[inside], mean=mu, sd=sigma1, log=TRUE)
  logphi2 <- dnorm(data[!inside], mean=mu, sd=sigma2, log=TRUE)
  
  loglik <- -length(data) * log(Z) + sum(logphi1) + sum(logphi2)
  return(loglik)
}

mle_int_param <- function(alpha, data, theta1_start = NULL, theta2_start = NULL) {
  if (is.null(theta1_start)) theta1_start <- quantile(data, alpha/2)
  if (is.null(theta2_start)) theta2_start <- quantile(data, 1-alpha/2)
  
  fit <- mle2(
    minuslogl = function(theta1, theta2) -loglik_int_param(theta1, theta2, alpha, data),
    start = list(theta1 = theta1_start, theta2 = theta2_start),
    method = "L-BFGS-B",
    lower = c(theta1 = min(data) - 10 * sd(data), theta2 = min(data) - 10 * sd(data)),
    upper = c(theta1 = max(data) + 10 * sd(data), theta2 = max(data) + 10 * sd(data))
  )
  return(fit)
}

set.seed(1)
# Parameters
alpha <- 1-0.95
mu1 <- -2
mu2 <- 3
n <- 1000

simdata <- hybrid_sampler(n, mu1, mu2, alpha)

fit <- mle_int_param(alpha, simdata)
summary(fit)

# Report the estimated min and max (order-independent)
theta1_hat <- coef(fit)[["theta1"]]
theta2_hat <- coef(fit)[["theta2"]]
L_hat <- min(theta1_hat, theta2_hat)
U_hat <- max(theta1_hat, theta2_hat)
cat(sprintf("Estimated min: %.4f, Estimated max: %.4f\n", L_hat, U_hat))


In [ ]:
# Visualize the log-likelihood surface for a given sample
mu1 <- -1; mu2 <- 1; alpha <- 0.01; n <- 50000
simdata <- hybrid_sampler(n, mu1, mu2, alpha)
L_seq <- seq(-2, 1, length=60)
U_seq <- seq(-1, 2, length=60)
# Assuming simdata, alpha, L_seq, U_seq already defined
ll_surface <- outer(L_seq, U_seq, Vectorize(function(L, U) 
  if (L < U) loglik_int_param(L, U, alpha, simdata) else NA))

filled.contour(L_seq, U_seq, ll_surface,
               xlab="L", ylab="U", main="Log-likelihood surface for (L, U)")
points(min(mu1, mu2), max(mu1, mu2), col=2, pch=19, cex=2)



In [ ]:
library(truncnorm)
library(nloptr)

# Negative log-likelihood: params = c(a, d), with d > 0, so L = a, U = a+d
loglik_int_param <- function(params, alpha, data) {
  if (any(is.na(params)) || any(!is.finite(params))) return(1e12)
  a <- params[1]; d <- params[2]
  if (d <= 1e-6) return(1e12) # enforce strict separation
  L <- a
  U <- a + d
  mu <- (L + U) / 2
  sigma1 <- sqrt(1/(2*alpha))
  sigma2 <- sqrt(1/(2*(1-alpha)))
  Phi1_L <- pnorm(L, mu, sigma1)
  Phi1_U <- pnorm(U, mu, sigma1)
  Phi2_L <- pnorm(L, mu, sigma2)
  Phi2_U <- pnorm(U, mu, sigma2)
  Z <- 1 - (Phi2_U - Phi2_L) + (Phi1_U - Phi1_L)
  if(!is.finite(Z) || Z <= 0) return(1e12)
  inside <- (data >= L) & (data <= U)
  logphi1 <- sum(dnorm(data[inside], mu, sigma1, log=TRUE))
  logphi2 <- sum(dnorm(data[!inside], mu, sigma2, log=TRUE))
  loglik <- -length(data) * log(Z) + logphi1 + logphi2
  if(!is.finite(loglik)) return(1e12)
  return(-loglik)
}

# The robust hybrid sampler (as before)
hybrid_sampler <- function(n, mu1, mu2, alpha) {
  mu <- (mu1 + mu2)/2
  L <- min(mu1, mu2)
  U <- max(mu1, mu2)
  sigma1 <- sqrt(1/(2*alpha))
  sigma2 <- sqrt(1/(2*(1-alpha)))
  Phi1_L <- pnorm(L, mu, sigma1)
  Phi1_U <- pnorm(U, mu, sigma1)
  Phi2_L <- pnorm(L, mu, sigma2)
  Phi2_U <- pnorm(U, mu, sigma2)
  A1 <- Phi1_U - Phi1_L
  A2 <- 1 - (Phi2_U - Phi2_L)
  prob1 <- A1 / (A1 + A2)
  B <- rbinom(n, 1, prob1)
  x <- numeric(n)
  if(sum(B==1) > 0)
    x[B==1] <- rtruncnorm(sum(B==1), L, U, mu, sigma1)
  if(sum(B==0) > 0) {
    idx0 <- which(B==0)
    tail_prob <- pnorm(L, mu, sigma2) / (pnorm(L, mu, sigma2) + 1 - pnorm(U, mu, sigma2))
    choose_left <- rbinom(length(idx0), 1, tail_prob)
    idx_left <- idx0[choose_left == 1]
    idx_right <- idx0[choose_left == 0]
    if(length(idx_left) > 0)
      x[idx_left] <- rtruncnorm(length(idx_left), -Inf, L, mu, sigma2)
    if(length(idx_right) > 0)
      x[idx_right] <- rtruncnorm(length(idx_right), U, Inf, mu, sigma2)
  }
  x
}

# Global + local optimizer with new parameterization
mle_int_param_overkill <- function(alpha, data) {
  # Use the actual data range to avoid CRS2 going wild
  datmin <- min(data); datmax <- max(data)
  datsd <- sd(data)
  lower <- c(datmin - 2 * datsd, 1e-3) # d > 0
  upper <- c(datmax + 2 * datsd, (datmax - datmin) + 4 * datsd)
  starts <- list(
    c(datmin, (datmax - datmin)/2),
    c(quantile(data, alpha/4), (quantile(data, 1-alpha/4) - quantile(data, alpha/4))),
    c(quantile(data, alpha/2), (quantile(data, 1-alpha/2) - quantile(data, alpha/2)))
  )
  best_val <- Inf
  best_sol <- NULL
  for (start in starts) {
    res <- tryCatch({
      nloptr(
        x0 = start,
        eval_f = function(params) loglik_int_param(params, alpha, data),
        lb = lower, ub = upper,
        opts = list(
          algorithm = "NLOPT_GN_CRS2_LM",
          maxeval = 3000,
          xtol_rel = 1e-8,
          ftol_rel = 1e-8,
          population = 40
        )
      )
    }, error = function(e) NULL)
    if (!is.null(res) && !is.null(res$objective) && is.finite(res$objective) && res$status > 0 && res$objective < best_val) {
      best_val <- res$objective
      best_sol <- res$solution
    }
  }
  if (is.null(best_sol)) stop("Global search failed for all starts!")
  res_local <- tryCatch({
    nloptr(
      x0 = best_sol,
      eval_f = function(params) loglik_int_param(params, alpha, data),
      lb = lower, ub = upper,
      opts = list(
        algorithm = "NLOPT_LN_COBYLA",
        maxeval = 1000,
        xtol_rel = 1e-8,
        ftol_rel = 1e-8
      )
    )
  }, error = function(e) NULL)
  if (is.null(res_local) || is.null(res_local$solution)) stop("Local polish failed!")
  ests <- res_local$solution
  L_hat <- ests[1]
  U_hat <- ests[1] + ests[2]
  return(c(est_L = L_hat, est_U = U_hat))
}

# Test run
set.seed(1)
alpha <- 1 - 0.95
mu1 <- -2
mu2 <- 2
n <- 2000

simdata <- hybrid_sampler(n, mu1, mu2, alpha)
fit <- mle_int_param_overkill(alpha, simdata)
cat(sprintf("Estimated min: %.4f, Estimated max: %.4f\n", fit["est_L"], fit["est_U"]))


In [ ]:
library(nloptr)
library(truncnorm)

# Negative log-likelihood: params = c(a, d), with d > 0, so L = a, U = a+d
loglik_int_param <- function(params, alpha, data) {
  if (any(is.na(params)) || any(!is.finite(params))) return(1e12)
  a <- params[1]; d <- params[2]
  if (d <= 1e-6) return(1e12) # enforce strict separation
  L <- a
  U <- a + d
  mu <- (L + U) / 2
  sigma1 <- sqrt(1/(2*alpha))
  sigma2 <- sqrt(1/(2*(1-alpha)))
  Phi1_L <- pnorm(L, mu, sigma1)
  Phi1_U <- pnorm(U, mu, sigma1)
  Phi2_L <- pnorm(L, mu, sigma2)
  Phi2_U <- pnorm(U, mu, sigma2)
  Z <- 1 - (Phi2_U - Phi2_L) + (Phi1_U - Phi1_L)
  if(!is.finite(Z) || Z <= 0) return(1e12)
  inside <- (data >= L) & (data <= U)
  logphi1 <- sum(dnorm(data[inside], mu, sigma1, log=TRUE))
  logphi2 <- sum(dnorm(data[!inside], mu, sigma2, log=TRUE))
  loglik <- -length(data) * log(Z) + logphi1 + logphi2
  if(!is.finite(loglik)) return(1e12)
  return(-loglik)
}

# Global + local optimizer with new parameterization
mle_int_param_overkill <- function(alpha, data) {
  datmin <- min(data); datmax <- max(data)
  datsd <- sd(data)
  lower <- c(datmin - 2 * datsd, 1e-3) # d > 0
  upper <- c(datmax + 2 * datsd, (datmax - datmin) + 4 * datsd)
  starts <- list(
    c(datmin, (datmax - datmin)/2),
    c(quantile(data, alpha/4), (quantile(data, 1-alpha/4) - quantile(data, alpha/4))),
    c(quantile(data, alpha/2), (quantile(data, 1-alpha/2) - quantile(data, alpha/2)))
  )
  best_val <- Inf
  best_sol <- NULL
  for (start in starts) {
    res <- tryCatch({
      nloptr(
        x0 = start,
        eval_f = function(params) loglik_int_param(params, alpha, data),
        lb = lower, ub = upper,
        opts = list(
          algorithm = "NLOPT_GN_CRS2_LM",
          maxeval = 3000,
          xtol_rel = 1e-8,
          ftol_rel = 1e-8,
          population = 40
        )
      )
    }, error = function(e) NULL)
    if (!is.null(res) && !is.null(res$objective) && is.finite(res$objective) && res$status > 0 && res$objective < best_val) {
      best_val <- res$objective
      best_sol <- res$solution
    }
  }
  if (is.null(best_sol)) stop("Global search failed for all starts!")
  res_local <- tryCatch({
    nloptr(
      x0 = best_sol,
      eval_f = function(params) loglik_int_param(params, alpha, data),
      lb = lower, ub = upper,
      opts = list(
        algorithm = "NLOPT_LN_COBYLA",
        maxeval = 1000,
        xtol_rel = 1e-8,
        ftol_rel = 1e-8
      )
    )
  }, error = function(e) NULL)
  if (is.null(res_local) || is.null(res_local$solution)) stop("Local polish failed!")
  ests <- res_local$solution
  L_hat <- ests[1]
  U_hat <- ests[1] + ests[2]
  return(c(est_L = L_hat, est_U = U_hat))
}

# ---- Main test ----

set.seed(42)
alpha <- 1-0.95
n <- 100000

# Sample standard normal data
data <- rnorm(n)

# Run MLE
fit <- mle_int_param_overkill(alpha, data)
cat(sprintf("Estimated L: %.4f, Estimated U: %.4f\n", fit["est_L"], fit["est_U"]))

# Optional: Compare to empirical quantiles
cat(sprintf("Empirical 2.5%%: %.4f, Empirical 97.5%%: %.4f\n",
            quantile(data, 0.025), quantile(data, 0.975)))


## Adding scale: $\tau^2$

In [ ]:
library(truncnorm)
library(ggplot2)

# Hybrid PDF with tau
hybrid_pdf <- function(x, mu1, mu2, alpha, tau = 1) {
  mu <- (mu1 + mu2) / 2
  L <- min(mu1, mu2)
  U <- max(mu1, mu2)
  sigma1 <- sqrt(tau^2/(2*alpha))
  sigma2 <- sqrt(tau^2/(2*(1-alpha)))
  Phi1 <- function(z) pnorm(z, mean=mu, sd=sigma1)
  Phi2 <- function(z) pnorm(z, mean=mu, sd=sigma2)
  A1 <- Phi1(U) - Phi1(L)
  A2 <- 1 - (Phi2(U) - Phi2(L))
  Z <- A1 + A2
  dens <- numeric(length(x))
  idx1 <- (x >= L) & (x <= U)
  dens[idx1] <- dnorm(x[idx1], mean=mu, sd=sigma1) / Z
  idx2 <- !idx1
  dens[idx2] <- dnorm(x[idx2], mean=mu, sd=sigma2) / Z
  return(dens)
}

# Hybrid sampler with tau
hybrid_sampler <- function(n, mu1, mu2, alpha, tau = 1) {
  mu <- (mu1 + mu2) / 2
  L <- min(mu1, mu2)
  U <- max(mu1, mu2)
  sigma1 <- sqrt(tau^2/(2*alpha))
  sigma2 <- sqrt(tau^2/(2*(1-alpha)))
  # For weights:
  Phi1_L <- pnorm(L, mean=mu, sd=sigma1)
  Phi1_U <- pnorm(U, mean=mu, sd=sigma1)
  Phi2_L <- pnorm(L, mean=mu, sd=sigma2)
  Phi2_U <- pnorm(U, mean=mu, sd=sigma2)
  A1 <- Phi1_U - Phi1_L
  A2 <- 1 - (Phi2_U - Phi2_L)
  Z <- A1 + A2
  prob1 <- A1 / Z
  # Bernoulli to decide regime
  B <- rbinom(n, size=1, prob=prob1)
  x <- numeric(n)
  # Inside [L, U]
  if(sum(B == 1) > 0)
    x[B == 1] <- rtruncnorm(sum(B == 1), a=L, b=U, mean=mu, sd=sigma1)
  # Outside [L, U]
  n2 <- sum(B == 0)
  if(n2 > 0) {
    left_tail_mass <- pnorm(L, mean=mu, sd=sigma2)
    right_tail_mass <- 1 - pnorm(U, mean=mu, sd=sigma2)
    total_tail_mass <- left_tail_mass + right_tail_mass
    idx_outside <- which(B == 0)
    choose_left <- rbinom(n2, size=1, prob=left_tail_mass / total_tail_mass)
    if(sum(choose_left) > 0)
      x[idx_outside[which(choose_left == 1)]] <- rtruncnorm(sum(choose_left), a=-Inf, b=L, mean=mu, sd=sigma2)
    if(sum(choose_left == 0) > 0)
      x[idx_outside[which(choose_left == 0)]] <- rtruncnorm(sum(choose_left == 0), a=U, b=Inf, mean=mu, sd=sigma2)
  }
  return(x)
}

# Set parameters
alpha <- 0.05
mu1 <- -3
mu2 <- 4
tau <- 10
set.seed(2024)
n <- 100000

samples <- hybrid_sampler(n, mu1, mu2, alpha, tau)
x_grid <- seq(mu1 - 4*tau, mu2 + 4*tau, length.out=1000)
dens_grid <- hybrid_pdf(x_grid, mu1, mu2, alpha, tau)
df <- data.frame(x = x_grid, dens = dens_grid)

ggplot() +
  geom_histogram(aes(x = samples, y = after_stat(density)), 
                 bins = 80, fill = "skyblue", alpha = 0.5, boundary = 0) +
  geom_line(data = df, aes(x = x, y = dens), color = "darkred", linewidth = 0.8) +
  labs(title = bquote("Interval (Hybrid) Normal,  "*tau^2*"="*.(tau^2)),
       x = "x", y = "Density") +
  theme_minimal(base_size = 15)


In [ ]:
library(truncnorm)
library(bbmle)

# --- Hybrid Sampler with tau^2 ---
hybrid_sampler <- function(n, mu1, mu2, alpha, tau = 1) {
  mu <- (mu1 + mu2) / 2
  L <- min(mu1, mu2)
  U <- max(mu1, mu2)
  sigma1 <- sqrt(tau^2/(2*alpha))
  sigma2 <- sqrt(tau^2/(2*(1-alpha)))
  Phi1_L <- pnorm(L, mean=mu, sd=sigma1)
  Phi1_U <- pnorm(U, mean=mu, sd=sigma1)
  Phi2_L <- pnorm(L, mean=mu, sd=sigma2)
  Phi2_U <- pnorm(U, mean=mu, sd=sigma2)
  A1 <- Phi1_U - Phi1_L
  A2 <- 1 - (Phi2_U - Phi2_L)
  Z <- A1 + A2
  prob1 <- A1 / Z
  B <- rbinom(n, size=1, prob=prob1)
  x <- numeric(n)
  if(sum(B == 1) > 0)
    x[B == 1] <- rtruncnorm(sum(B == 1), a=L, b=U, mean=mu, sd=sigma1)
  n2 <- sum(B == 0)
  if(n2 > 0) {
    left_tail_mass <- pnorm(L, mean=mu, sd=sigma2)
    right_tail_mass <- 1 - pnorm(U, mean=mu, sd=sigma2)
    total_tail_mass <- left_tail_mass + right_tail_mass
    idx_outside <- which(B == 0)
    choose_left <- rbinom(n2, size=1, prob=left_tail_mass / total_tail_mass)
    if(sum(choose_left) > 0)
      x[idx_outside[which(choose_left == 1)]] <- rtruncnorm(sum(choose_left), a=-Inf, b=L, mean=mu, sd=sigma2)
    if(sum(choose_left == 0) > 0)
      x[idx_outside[which(choose_left == 0)]] <- rtruncnorm(sum(choose_left == 0), a=U, b=Inf, mean=mu, sd=sigma2)
  }
  return(x)
}

# --- Log-likelihood for hybrid/interval model with tau^2 ---
loglik_int_param <- function(mu1, mu2, log_tau, alpha, data) {
  tau <- exp(log_tau) # always positive
  mu <- (mu1 + mu2) / 2
  L <- min(mu1, mu2)
  U <- max(mu1, mu2)
  if (L >= U) return(-1e10) # enforce L < U
  sigma1 <- sqrt(tau^2 / (2*alpha))
  sigma2 <- sqrt(tau^2 / (2*(1-alpha)))
  Phi1_L <- pnorm(L, mean=mu, sd=sigma1)
  Phi1_U <- pnorm(U, mean=mu, sd=sigma1)
  Phi2_L <- pnorm(L, mean=mu, sd=sigma2)
  Phi2_U <- pnorm(U, mean=mu, sd=sigma2)
  Z <- 1 - (Phi2_U - Phi2_L) + (Phi1_U - Phi1_L)
  inside <- (data >= L) & (data <= U)
  logphi1 <- dnorm(data[inside], mean=mu, sd=sigma1, log=TRUE)
  logphi2 <- dnorm(data[!inside], mean=mu, sd=sigma2, log=TRUE)
  loglik <- -length(data) * log(Z) + sum(logphi1) + sum(logphi2)
  return(loglik)
}

# --- MLE routine for interval/hybrid with tau^2 ---
mle_int_param <- function(alpha, data, mu1_start = NULL, mu2_start = NULL, tau_start = NULL) {
  if (is.null(mu1_start)) mu1_start <- quantile(data, alpha/2)
  if (is.null(mu2_start)) mu2_start <- quantile(data, 1 - alpha/2)
  if (is.null(tau_start)) tau_start <- sd(data) # Reasonable guess
  fit <- mle2(
    minuslogl = function(mu1, mu2, log_tau) -loglik_int_param(mu1, mu2, log_tau, alpha, data),
    start = list(mu1 = mu1_start, mu2 = mu2_start, log_tau = log(tau_start)),
    method = "L-BFGS-B",
    lower = c(mu1 = min(data) - 5*sd(data), mu2 = min(data) - 5*sd(data), log_tau = log(0.05*sd(data))),
    upper = c(mu1 = max(data) + 5*sd(data), mu2 = max(data) + 5*sd(data), log_tau = log(20*sd(data)))
  )
  return(fit)
}

# --- Simulate and fit ---
set.seed(1)
alpha <- 1 - 0.95
mu1 <- -2
mu2 <- 3
tau <- 1
n <- 1000

simdata <- hybrid_sampler(n, mu1, mu2, alpha, tau)

fit <- mle_int_param(alpha, simdata)
print(summary(fit))

# Report the estimated mus and tau (back-transform tau)
theta_hat <- coef(fit)
mu1_hat <- theta_hat[["mu1"]]
mu2_hat <- theta_hat[["mu2"]]
tau_hat <- exp(theta_hat[["log_tau"]])
cat(sprintf("Estimated mu1: %.4f, Estimated mu2: %.4f, Estimated tau: %.4f\n",
            mu1_hat, mu2_hat, tau_hat))


In [ ]:
library(truncnorm)
library(nloptr)

# Negative log-likelihood with tau^2 scale parameter
# params = c(a, d, log_tau): L = a, U = a+d, tau = exp(log_tau)
loglik_int_param <- function(params, alpha, data) {
  if (any(is.na(params)) || any(!is.finite(params))) return(1e12)
  a <- params[1]; d <- params[2]; log_tau <- params[3]
  if (d <= 1e-6) return(1e12) # enforce strict separation
  tau <- exp(log_tau)
  L <- a
  U <- a + d
  mu <- (L + U) / 2
  sigma1 <- sqrt(tau^2/(2*alpha))
  sigma2 <- sqrt(tau^2/(2*(1-alpha)))
  Phi1_L <- pnorm(L, mu, sigma1)
  Phi1_U <- pnorm(U, mu, sigma1)
  Phi2_L <- pnorm(L, mu, sigma2)
  Phi2_U <- pnorm(U, mu, sigma2)
  Z <- 1 - (Phi2_U - Phi2_L) + (Phi1_U - Phi1_L)
  if(!is.finite(Z) || Z <= 0) return(1e12)
  inside <- (data >= L) & (data <= U)
  logphi1 <- sum(dnorm(data[inside], mu, sigma1, log=TRUE))
  logphi2 <- sum(dnorm(data[!inside], mu, sigma2, log=TRUE))
  loglik <- -length(data) * log(Z) + logphi1 + logphi2
  if(!is.finite(loglik)) return(1e12)
  return(-loglik)
}

# Hybrid sampler with tau^2
hybrid_sampler <- function(n, mu1, mu2, alpha, tau = 1) {
  mu <- (mu1 + mu2)/2
  L <- min(mu1, mu2)
  U <- max(mu1, mu2)
  sigma1 <- sqrt(tau^2/(2*alpha))
  sigma2 <- sqrt(tau^2/(2*(1-alpha)))
  Phi1_L <- pnorm(L, mu, sigma1)
  Phi1_U <- pnorm(U, mu, sigma1)
  Phi2_L <- pnorm(L, mu, sigma2)
  Phi2_U <- pnorm(U, mu, sigma2)
  A1 <- Phi1_U - Phi1_L
  A2 <- 1 - (Phi2_U - Phi2_L)
  prob1 <- A1 / (A1 + A2)
  B <- rbinom(n, 1, prob1)
  x <- numeric(n)
  if(sum(B==1) > 0)
    x[B==1] <- rtruncnorm(sum(B==1), L, U, mu, sigma1)
  if(sum(B==0) > 0) {
    idx0 <- which(B==0)
    tail_prob <- pnorm(L, mu, sigma2) / (pnorm(L, mu, sigma2) + 1 - pnorm(U, mu, sigma2))
    choose_left <- rbinom(length(idx0), 1, tail_prob)
    idx_left <- idx0[choose_left == 1]
    idx_right <- idx0[choose_left == 0]
    if(length(idx_left) > 0)
      x[idx_left] <- rtruncnorm(length(idx_left), -Inf, L, mu, sigma2)
    if(length(idx_right) > 0)
      x[idx_right] <- rtruncnorm(length(idx_right), U, Inf, mu, sigma2)
  }
  x
}

# Global + local optimizer with parameterization (a, d, tau)
mle_int_param_overkill <- function(alpha, data) {
  datmin <- min(data); datmax <- max(data)
  datsd <- sd(data)
  lower <- c(datmin - 10*datsd, 1e-6, log(0.0005*datsd))
  upper <- c(datmax + 10*datsd, (datmax-datmin)+10*datsd, log(2000*datsd))
  starts <- list(
    c(datmin, (datmax-datmin)/2, log(datsd)),
    c(quantile(data, alpha/4), (quantile(data, 1-alpha/4) - quantile(data, alpha/4)), log(datsd)),
    c(quantile(data, alpha/2), (quantile(data, 1-alpha/2) - quantile(data, alpha/2)), log(datsd))
  )
  best_val <- Inf
  best_sol <- NULL
  for (start in starts) {
    res <- tryCatch({
      nloptr(
        x0 = start,
        eval_f = function(params) loglik_int_param(params, alpha, data),
        lb = lower, ub = upper,
        opts = list(
          algorithm = "NLOPT_GN_CRS2_LM",
          maxeval = 3000,
          xtol_rel = 1e-8,
          ftol_rel = 1e-8,
          population = 40
        )
      )
    }, error = function(e) NULL)
    if (!is.null(res) && !is.null(res$objective) && is.finite(res$objective) && res$status > 0 && res$objective < best_val) {
      best_val <- res$objective
      best_sol <- res$solution
    }
  }
  if (is.null(best_sol)) stop("Global search failed for all starts!")
  res_local <- tryCatch({
    nloptr(
      x0 = best_sol,
      eval_f = function(params) loglik_int_param(params, alpha, data),
      lb = lower, ub = upper,
      opts = list(
        algorithm = "NLOPT_LN_COBYLA",
        maxeval = 1000,
        xtol_rel = 1e-8,
        ftol_rel = 1e-8
      )
    )
  }, error = function(e) NULL)
  if (is.null(res_local) || is.null(res_local$solution)) stop("Local polish failed!")
  ests <- res_local$solution
  L_hat <- ests[1]
  U_hat <- ests[1] + ests[2]
  tau_hat <- exp(ests[3])
  return(c(est_L = L_hat, est_U = U_hat, est_tau = tau_hat))
}

# --- Test run with tau^2 as a parameter ---
set.seed(1)
alpha <- 1 - 0.99
mu1 <- -3
mu2 <- 4
tau <- 2
n <- 1000

simdata <- hybrid_sampler(n, mu1, mu2, alpha, tau)
fit <- mle_int_param_overkill(alpha, simdata)
cat(sprintf("Estimated min: %.4f, Estimated max: %.4f, Estimated tau: %.4f\n",
            fit["est_L"], fit["est_U"], fit["est_tau"]))


In [ ]:
# set.seed(42)
alpha <- 1-0.95
n <- 5000

simdata <- rnorm(n)
# Run MLE
fit <- mle_int_param_overkill(alpha, simdata)
cat(sprintf("Estimated L: %.4f, Estimated U: %.4f, Estimated tau: %.4f\n", fit["est_L"], fit["est_U"], fit["est_tau"]))

# Optional: Compare to empirical quantiles
cat(sprintf("Empirical Lower: %.4f, Empirical Upper: %.4f\n",
            quantile(simdata, alpha/2), quantile(simdata, 1-alpha/2)))
